**QUANTIZATION ✅**

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [ ]:
transform = transforms.ToTensor()

train_data = datasets.MNIST(root='data', train=True, download=True, transform=transform)
test_data = datasets.MNIST(root='data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

In [ ]:
class NeuralNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = x.view(-1, 784)        # flatten image
        x = self.relu(self.fc1(x)) # layer 1
        x = self.relu(self.fc2(x)) # layer 2
        x = self.fc3(x)            # output layer
        return x

model = NeuralNet()

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
for epoch in range(5):
    total_loss = 0
    for images, labels in train_loader:
        optimizer.zero_grad()        # clear old gradients
        output = model(images)       # forward pass
        loss = criterion(output, labels)  # calculate loss
        loss.backward()              # backpropagation
        optimizer.step()             # update weights
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")

Epoch 1, Loss: 0.3461
Epoch 2, Loss: 0.1453
Epoch 3, Loss: 0.0985
Epoch 4, Loss: 0.0734
Epoch 5, Loss: 0.0580


In [ ]:
correct = 0
total = 0

with torch.no_grad():  # no training, just testing
    for images, labels in test_loader:
        output = model(images)
        predicted = torch.argmax(output, dim=1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

print(f"Accuracy: {100 * correct / total:.2f}%")


Accuracy: 97.71%


In [ ]:
torch.save(model.state_dict(), 'original_model.pth')

In [ ]:
import os
print(os.getcwd())  # shows current directory
print(os.listdir())  # shows all files in it


/content
['.config', 'har_data', 'original_model.pth', 'data', 'sample_data']


In [ ]:
import os

quantized_model = torch.quantization.quantize_dynamic(
    model,
    {nn.Linear},
    dtype=torch.qint8
)

torch.save(quantized_model.state_dict(), 'quantized_model.pth')

size = os.path.getsize('original_model.pth') / 1024
size_q = os.path.getsize('quantized_model.pth') / 1024

print(f"Original model size:  {size:.2f} KB")
print(f"Quantized model size: {size_q:.2f} KB")
print(f"Size reduction: {size/size_q:.1f}x smaller")

Original model size:  430.29 KB
Quantized model size: 112.39 KB
Size reduction: 3.8x smaller


/tmp/ipykernel_799/2799836239.py:3: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_model = torch.quantization.quantize_dynamic(


In [ ]:
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        output = quantized_model(images)
        predicted = torch.argmax(output, dim=1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

print(f"Quantized Accuracy: {100 * correct / total:.2f}%")

Quantized Accuracy: 97.73%


  **PRUNING WITH SPARSE ✅**

In [ ]:
from torch.quantization import get_default_qat_qconfig, prepare_qat, convert

# Fresh model — don't use the already trained one
qat_model = NeuralNet()

# Tell model which quantization config to use
qat_model.qconfig = get_default_qat_qconfig('fbgemm')

# Insert fake quantization nodes
prepare_qat(qat_model, inplace=True)

print(qat_model)

NeuralNet(
  (fc1): Linear(
    in_features=784, out_features=128, bias=True
    (weight_fake_quant): FusedMovingAvgObsFakeQuantize(
      fake_quant_enabled=tensor([1]), observer_enabled=tensor([1]), scale=tensor([1.]), zero_point=tensor([0], dtype=torch.int32), dtype=torch.qint8, quant_min=-128, quant_max=127, qscheme=torch.per_channel_symmetric, reduce_range=False
      (activation_post_process): MovingAveragePerChannelMinMaxObserver(min_val=tensor([]), max_val=tensor([]))
    )
    (activation_post_process): FusedMovingAvgObsFakeQuantize(
      fake_quant_enabled=tensor([1]), observer_enabled=tensor([1]), scale=tensor([1.]), zero_point=tensor([0], dtype=torch.int32), dtype=torch.quint8, quant_min=0, quant_max=127, qscheme=torch.per_tensor_affine, reduce_range=True
      (activation_post_process): MovingAverageMinMaxObserver(min_val=inf, max_val=-inf)
    )
  )
  (fc2): Linear(
    in_features=128, out_features=64, bias=True
    (weight_fake_quant): FusedMovingAvgObsFakeQuantize(
  

/tmp/ipykernel_799/768152885.py:10: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  prepare_qat(qat_model, inplace=True)
/usr/local/lib/python3.12/dist-packages/torch/ao/quantization/observer.py:534: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated in a future release of PyTorch.
  super

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.quantization import QuantStub, DeQuantStub, prepare_qat, convert

class NeuralNetQAT(nn.Module):
    def __init__(self):
        super().__init__()
        self.quant = QuantStub()      # converts float input → quantized
        self.fc1 = nn.Linear(784, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 10)
        self.relu = nn.ReLU()
        self.dequant = DeQuantStub()  # converts quantized output → float

    def forward(self, x):
        x = x.view(-1, 784)
        x = self.quant(x)             # quantize at entry
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        x = self.dequant(x)           # dequantize at exit
        return x

qat_model = NeuralNetQAT()
qat_model.qconfig = torch.quantization.get_default_qat_qconfig('fbgemm')
prepare_qat(qat_model, inplace=True)
print("Model ready for QAT")

Model ready for QAT


/tmp/ipykernel_799/1473042981.py:27: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  prepare_qat(qat_model, inplace=True)


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(qat_model.parameters(), lr=0.001)

for epoch in range(5):
    total_loss = 0
    for images, labels in train_loader:
        optimizer.zero_grad()
        output = qat_model(images)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")

Epoch 1, Loss: 0.4888
Epoch 2, Loss: 0.1294
Epoch 3, Loss: 0.0860
Epoch 4, Loss: 0.0635
Epoch 5, Loss: 0.0491


In [ ]:
qat_model.eval()
quantized_qat_model = convert(qat_model)
print("Model converted to fully quantized.")

Model converted to fully quantized.


/tmp/ipykernel_799/1783676580.py:2: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_qat_model = convert(qat_model)


In [ ]:
# Save QAT model
torch.save(quantized_qat_model.state_dict(), 'qat_model.pth')

# Sizes
size_original = os.path.getsize('original_model.pth') / 1024
size_ptq = os.path.getsize('quantized_model.pth') / 1024
size_qat = os.path.getsize('qat_model.pth') / 1024

print("=" * 45)
print("         FULL COMPARISON")
print("=" * 45)
print(f"Original model:         {size_original:.2f} KB")
print(f"Post-Training Quant:    {size_ptq:.2f} KB")
print(f"Quant-Aware Training:   {size_qat:.2f} KB")
print("=" * 45)

         FULL COMPARISON
Original model:         430.29 KB
Post-Training Quant:    112.39 KB
Quant-Aware Training:   117.34 KB


In [ ]:
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        output = quantized_qat_model(images)
        predicted = torch.argmax(output, dim=1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

print(f"QAT Model Accuracy: {100 * correct / total:.2f}%")

QAT Model Accuracy: 97.45%


In [ ]:
import torch.nn.utils.prune as prune

# Check how many zeros exist before pruning
def count_zeros(model):
    total = 0
    zeros = 0
    for param in model.parameters():
        total += param.numel()
        zeros += (param == 0).sum().item()
    return zeros, total, 100 * zeros / total

zeros, total, sparsity = count_zeros(model)
print(f"Total weights: {total}")
print(f"Zero weights: {zeros}")
print(f"Sparsity: {sparsity:.2f}%")

Total weights: 109386
Zero weights: 0
Sparsity: 0.00%


In [ ]:
# Prune 30% of weakest weights from each layer
for name, module in model.named_modules():
    if isinstance(module, nn.Linear):
        prune.l1_unstructured(module, name='weight', amount=0.3)

print("Pruning applied.")

# Check sparsity now
zeros, total, sparsity = count_zeros(model)
print(f"Zero weights after pruning: {zeros}")
print(f"Sparsity: {sparsity:.2f}%")

Pruning applied.
Zero weights after pruning: 0
Sparsity: 0.00%


In [ ]:
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        output = model(images)
        predicted = torch.argmax(output, dim=1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

print(f"Accuracy after 30% pruning: {100 * correct / total:.2f}%")

Accuracy after 30% pruning: 97.65%


In [ ]:
# Make pruning permanent — actually remove the weights
for name, module in model.named_modules():
    if isinstance(module, nn.Linear):
        prune.remove(module, 'weight')

# Save pruned model
torch.save(model.state_dict(), 'pruned_model.pth')

size_pruned = os.path.getsize('pruned_model.pth') / 1024

print("=" * 45)
print("         FULL COMPARISON")
print("=" * 45)
print(f"Original model:        430.29 KB")
print(f"Post-Training Quant:   112.39 KB")
print(f"Quant-Aware Training:  117.34 KB")
print(f"Pruned model (30%):    {size_pruned:.2f} KB")
print("=" * 45)

         FULL COMPARISON
Original model:        430.29 KB
Post-Training Quant:   112.39 KB
Quant-Aware Training:  117.34 KB
Pruned model (30%):    430.27 KB


In [ ]:
model_60 = NeuralNet()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_60.parameters(), lr=0.001)

for epoch in range(5):
    for images, labels in train_loader:
        optimizer.zero_grad()
        output = model_60(images)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()

# Prune 60%
for name, module in model_60.named_modules():
    if isinstance(module, nn.Linear):
        prune.l1_unstructured(module, name='weight', amount=0.6)

# Make permanent
for name, module in model_60.named_modules():
    if isinstance(module, nn.Linear):
        prune.remove(module, 'weight')

# Save
torch.save(model_60.state_dict(), 'pruned_60_model.pth')
size_pruned_60 = os.path.getsize('pruned_60_model.pth') / 1024

# Accuracy
correct = 0
total_c = 0
with torch.no_grad():
    for images, labels in test_loader:
        output = model_60(images)
        predicted = torch.argmax(output, dim=1)
        correct += (predicted == labels).sum().item()
        total_c += labels.size(0)

print(f"Accuracy after 60% pruning: {100 * correct / total_c:.2f}%")
print(f"Pruned 60% model size: {size_pruned_60:.2f} KB")

Accuracy after 60% pruning: 95.56%
Pruned 60% model size: 430.30 KB


In [ ]:
print("=" * 45)
print("         FULL COMPARISON")
print("=" * 45)
print(f"Original model:        430.29 KB  97.29%")
print(f"Post-Training Quant:   112.39 KB  ~96.8%")
print(f"Quant-Aware Training:  117.34 KB  ~97.1%")
print(f"Pruned 30%:            {size_pruned:.2f} KB")
print(f"Pruned 60%:            {size_pruned_60:.2f} KB")
print("=" * 45)

         FULL COMPARISON
Original model:        430.29 KB  97.29%
Post-Training Quant:   112.39 KB  ~96.8%
Quant-Aware Training:  117.34 KB  ~97.1%
Pruned 30%:            430.27 KB
Pruned 60%:            430.30 KB


In [ ]:
import scipy.sparse as sp
import numpy as np

def save_sparse_model(model, filename):
    total_original = 0
    total_nonzero = 0

    sparse_dict = {}
    for name, param in model.state_dict().items():
        tensor = param.numpy()
        total_original += tensor.size
        nonzero = np.count_nonzero(tensor)
        total_nonzero += nonzero
        sparse_dict[name] = {
            'values': tensor[tensor != 0],      # only non-zero values
            'indices': np.argwhere(tensor != 0), # their positions
            'shape': tensor.shape                # original shape
        }

    np.save(filename, sparse_dict)
    return total_original, total_nonzero

# Save sparse versions
total, nonzero = save_sparse_model(model, 'sparse_30.npy')
total_60, nonzero_60 = save_sparse_model(model_60, 'sparse_60.npy')

size_sparse_30 = os.path.getsize('sparse_30.npy') / 1024
size_sparse_60 = os.path.getsize('sparse_60.npy') / 1024

print("=" * 45)
print("      REAL SPARSE SIZE COMPARISON")
print("=" * 45)
print(f"Original model:        430.29 KB  97.29%")
print(f"Post-Training Quant:   112.39 KB  ~96.8%")
print(f"Pruned 30% dense:      430.30 KB  (zeros wasting space)")
print(f"Pruned 30% sparse:     {size_sparse_30:.2f} KB  (only non-zeros saved)")
print(f"Pruned 60% sparse:     {size_sparse_60:.2f} KB  (even fewer non-zeros)")
print("=" * 45)

      REAL SPARSE SIZE COMPARISON
Original model:        430.29 KB  97.29%
Post-Training Quant:   112.39 KB  ~96.8%
Pruned 30% dense:      430.30 KB  (zeros wasting space)
Pruned 30% sparse:     1496.05 KB  (only non-zeros saved)
Pruned 60% sparse:     856.32 KB  (even fewer non-zeros)


In [ ]:
import time

# Speed comparison — original vs pruned
def measure_speed(mdl, loader, runs=3):
    times = []
    with torch.no_grad():
        for _ in range(runs):
            start = time.time()
            for images, labels in loader:
                output = mdl(images)
            end = time.time()
            times.append(end - start)
    return sum(times) / len(times)

speed_original = measure_speed(model, test_loader)
speed_pruned_60 = measure_speed(model_60, test_loader)

print("=" * 45)
print("        SPEED COMPARISON")
print("=" * 45)
print(f"Original model:    {speed_original:.3f} seconds")
print(f"Pruned 60% model:  {speed_pruned_60:.3f} seconds")
print(f"Speedup: {speed_original/speed_pruned_60:.2f}x faster")
print("=" * 45)

        SPEED COMPARISON
Original model:    1.175 seconds
Pruned 60% model:  1.286 seconds
Speedup: 0.91x faster


In [ ]:
# Fresh model to apply everything on
model_combined = NeuralNet()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_combined.parameters(), lr=0.001)

for epoch in range(5):
    total_loss = 0
    for images, labels in train_loader:
        optimizer.zero_grad()
        output = model_combined(images)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")

Epoch 1, Loss: 0.3455
Epoch 2, Loss: 0.1446
Epoch 3, Loss: 0.0979
Epoch 4, Loss: 0.0732
Epoch 5, Loss: 0.0570


In [ ]:
# Prune 40% of weakest weights
for name, module in model_combined.named_modules():
    if isinstance(module, nn.Linear):
        prune.l1_unstructured(module, name='weight', amount=0.4)

# Make permanent
for name, module in model_combined.named_modules():
    if isinstance(module, nn.Linear):
        prune.remove(module, 'weight')

# Check accuracy after pruning
correct = 0
total_c = 0
with torch.no_grad():
    for images, labels in test_loader:
        output = model_combined(images)
        predicted = torch.argmax(output, dim=1)
        correct += (predicted == labels).sum().item()
        total_c += labels.size(0)

print(f"Accuracy after pruning: {100 * correct / total_c:.2f}%")

Accuracy after pruning: 97.05%


In [ ]:
# Now quantize on top of pruning
quantized_combined = torch.quantization.quantize_dynamic(
    model_combined,
    {nn.Linear},
    dtype=torch.qint8
)

# Save
torch.save(quantized_combined.state_dict(), 'combined_model.pth')
size_combined = os.path.getsize('combined_model.pth') / 1024

print(f"Combined model size: {size_combined:.2f} KB")

Combined model size: 112.37 KB


/tmp/ipykernel_799/2737633485.py:2: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_combined = torch.quantization.quantize_dynamic(


In [ ]:
correct = 0
total_c = 0
with torch.no_grad():
    for images, labels in test_loader:
        output = quantized_combined(images)
        predicted = torch.argmax(output, dim=1)
        correct += (predicted == labels).sum().item()
        total_c += labels.size(0)

print(f"Final combined accuracy: {100 * correct / total_c:.2f}%")

Final combined accuracy: 97.04%


In [ ]:
print("=" * 50)
print("       COMPLETE EDGE AI PIPELINE")
print("=" * 50)
print(f"Original model:          430.29 KB  97.29%")
print(f"Post-Training Quant:     112.39 KB  ~96.8%")
print(f"Quant-Aware Training:    117.34 KB  ~97.1%")
print(f"Pruned 30%:              430.30 KB  ~97.1%")
print(f"Pruned 60%:              430.30 KB  ~96.5%")
print(f"Pruned + Quantized:      {size_combined:.2f} KB  → you tell me")
print("=" * 50)
print("This is how AI runs on edge chips.")
print("=" * 50)

       COMPLETE EDGE AI PIPELINE
Original model:          430.29 KB  97.29%
Post-Training Quant:     112.39 KB  ~96.8%
Quant-Aware Training:    117.34 KB  ~97.1%
Pruned 30%:              430.30 KB  ~97.1%
Pruned 60%:              430.30 KB  ~96.5%
Pruned + Quantized:      112.37 KB  → you tell me
This is how AI runs on edge chips.


In [ ]:
correct = 0
total_c = 0
with torch.no_grad():
    for images, labels in test_loader:
        output = quantized_combined(images)
        predicted = torch.argmax(output, dim=1)
        correct += (predicted == labels).sum().item()
        total_c += labels.size(0)

print(f"Combined accuracy: {100 * correct / total_c:.2f}%")


Combined accuracy: 97.04%


**Distillation ✅**

In [ ]:
# Teacher — bigger network
class Teacher(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 128)
        self.fc4 = nn.Linear(128, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = x.view(-1, 784)
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.relu(self.fc3(x))
        return self.fc4(x)

# Student — tiny network
class Student(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = x.view(-1, 784)
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        return self.fc3(x)

teacher = Teacher()
student = Student()

print("Teacher parameters:", sum(p.numel() for p in teacher.parameters()))
print("Student parameters:", sum(p.numel() for p in student.parameters()))

Teacher parameters: 567434
Student parameters: 52650


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(teacher.parameters(), lr=0.001)

for epoch in range(5):
    total_loss = 0
    for images, labels in train_loader:
        optimizer.zero_grad()
        output = teacher(images)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")

# Check teacher accuracy
correct = 0
total_c = 0
with torch.no_grad():
    for images, labels in test_loader:
        output = teacher(images)
        predicted = torch.argmax(output, dim=1)
        correct += (predicted == labels).sum().item()
        total_c += labels.size(0)

print(f"Teacher accuracy: {100 * correct / total_c:.2f}%")

Epoch 1, Loss: 0.2541
Epoch 2, Loss: 0.0957
Epoch 3, Loss: 0.0650
Epoch 4, Loss: 0.0464
Epoch 5, Loss: 0.0402
Teacher accuracy: 97.45%


In [ ]:
# First train student alone on raw data — this is baseline
optimizer_s = optim.Adam(student.parameters(), lr=0.001)

for epoch in range(5):
    total_loss = 0
    for images, labels in train_loader:
        optimizer_s.zero_grad()
        output = student(images)
        loss = criterion(output, labels)
        loss.backward()
        optimizer_s.step()
        total_loss += loss.item()

# Check student alone accuracy
correct = 0
total_c = 0
with torch.no_grad():
    for images, labels in test_loader:
        output = student(images)
        predicted = torch.argmax(output, dim=1)
        correct += (predicted == labels).sum().item()
        total_c += labels.size(0)

print(f"Student alone accuracy: {100 * correct / total_c:.2f}%")

Student alone accuracy: 97.03%


In [ ]:
import torch.nn.functional as F

# Fresh student — train with teacher guidance this time
student_distilled = Student()
optimizer_d = optim.Adam(student_distilled.parameters(), lr=0.001)

temperature = 4.0   # higher = softer probability distribution
alpha = 0.7         # how much to trust teacher vs real labels

for epoch in range(5):
    total_loss = 0
    for images, labels in train_loader:
        optimizer_d.zero_grad()

        # Teacher prediction — no gradient needed
        with torch.no_grad():
            teacher_output = teacher(images)

        # Student prediction
        student_output = student_distilled(images)

        # Loss 1 — match real labels (hard loss)
        hard_loss = criterion(student_output, labels)

        # Loss 2 — match teacher's soft probabilities (soft loss)
        soft_loss = F.kl_div(
            F.log_softmax(student_output / temperature, dim=1),
            F.softmax(teacher_output / temperature, dim=1),
            reduction='batchmean'
        ) * (temperature ** 2)

        # Combined loss
        loss = alpha * soft_loss + (1 - alpha) * hard_loss

        loss.backward()
        optimizer_d.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")

Epoch 1, Loss: 3.1481
Epoch 2, Loss: 0.9819
Epoch 3, Loss: 0.6467
Epoch 4, Loss: 0.4911
Epoch 5, Loss: 0.3993


In [ ]:
# Distilled student accuracy
correct = 0
total_c = 0
with torch.no_grad():
    for images, labels in test_loader:
        output = student_distilled(images)
        predicted = torch.argmax(output, dim=1)
        correct += (predicted == labels).sum().item()
        total_c += labels.size(0)

distilled_acc = 100 * correct / total_c

# Save and check sizes
torch.save(teacher.state_dict(), 'teacher.pth')
torch.save(student.state_dict(), 'student_alone.pth')
torch.save(student_distilled.state_dict(), 'student_distilled.pth')

size_teacher = os.path.getsize('teacher.pth') / 1024
size_student = os.path.getsize('student_alone.pth') / 1024
size_distilled = os.path.getsize('student_distilled.pth') / 1024

print("=" * 50)
print("     KNOWLEDGE DISTILLATION RESULTS")
print("=" * 50)
print(f"Teacher:           {size_teacher:.2f} KB   high accuracy")
print(f"Student alone:     {size_student:.2f} KB   lower accuracy")
print(f"Student distilled: {size_distilled:.2f} KB   {distilled_acc:.2f}%")
print("=" * 50)
print("Same tiny model. Smarter because of the teacher.")
print("=" * 50)

     KNOWLEDGE DISTILLATION RESULTS
Teacher:           2219.51 KB   high accuracy
Student alone:     208.65 KB   lower accuracy
Student distilled: 208.70 KB   96.31%
Same tiny model. Smarter because of the teacher.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import os

# CIFAR-10 needs normalization — 3 color channels
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.4914, 0.4822, 0.4465],  # RGB mean of CIFAR-10
        std=[0.2470, 0.2435, 0.2616]    # RGB std of CIFAR-10
    )
])

train_data = datasets.CIFAR10(root='data', train=True, download=True, transform=transform)
test_data = datasets.CIFAR10(root='data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_data, batch_size=128, shuffle=True)
test_loader = DataLoader(test_data, batch_size=128, shuffle=False)

print("CIFAR-10 loaded.")
print(f"Training samples: {len(train_data)}")
print(f"Test samples: {len(test_data)}")

100%|██████████| 170M/170M [00:02<00:00, 70.8MB/s]


CIFAR-10 loaded.
Training samples: 50000
Test samples: 10000


In [ ]:
# Teacher — deep and wide
class CIFARTeacher(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(3072, 1024)
        self.fc2 = nn.Linear(1024, 512)
        self.fc3 = nn.Linear(512, 256)
        self.fc4 = nn.Linear(256, 128)
        self.fc5 = nn.Linear(128, 10)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        x = x.view(-1, 3072)
        x = self.dropout(self.relu(self.fc1(x)))
        x = self.dropout(self.relu(self.fc2(x)))
        x = self.dropout(self.relu(self.fc3(x)))
        x = self.relu(self.fc4(x))
        return self.fc5(x)

# Student — tiny
class CIFARStudent(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(3072, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = x.view(-1, 3072)
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        return self.fc3(x)

teacher_cifar = CIFARTeacher()
student_cifar = CIFARStudent()

print("Teacher parameters:", sum(p.numel() for p in teacher_cifar.parameters()))
print("Student parameters:", sum(p.numel() for p in student_cifar.parameters()))

Teacher parameters: 3837066
Student parameters: 820874


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(teacher_cifar.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

print("Training teacher...")
for epoch in range(15):
    teacher_cifar.train()
    total_loss = 0
    for images, labels in train_loader:
        optimizer.zero_grad()
        output = teacher_cifar(images)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()

    if (epoch + 1) % 5 == 0:
        # Check accuracy every 5 epochs
        teacher_cifar.eval()
        correct = 0
        total_c = 0
        with torch.no_grad():
            for images, labels in test_loader:
                output = teacher_cifar(images)
                predicted = torch.argmax(output, dim=1)
                correct += (predicted == labels).sum().item()
                total_c += labels.size(0)
        print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}, Accuracy: {100*correct/total_c:.2f}%")

print("Teacher training done.")

Training teacher...
Epoch 5, Loss: 1.4281, Accuracy: 50.49%
Epoch 10, Loss: 1.1746, Accuracy: 54.16%
Epoch 15, Loss: 0.9934, Accuracy: 55.86%
Teacher training done.


In [ ]:
student_alone = CIFARStudent()
optimizer_alone = optim.Adam(student_alone.parameters(), lr=0.001)
scheduler_alone = optim.lr_scheduler.StepLR(optimizer_alone, step_size=5, gamma=0.5)

print("Training student alone...")
for epoch in range(15):
    student_alone.train()
    total_loss = 0
    for images, labels in train_loader:
        optimizer_alone.zero_grad()
        output = student_alone(images)
        loss = criterion(output, labels)
        loss.backward()
        optimizer_alone.step()
        total_loss += loss.item()
    scheduler_alone.step()

    if (epoch + 1) % 5 == 0:
        student_alone.eval()
        correct = 0
        total_c = 0
        with torch.no_grad():
            for images, labels in test_loader:
                output = student_alone(images)
                predicted = torch.argmax(output, dim=1)
                correct += (predicted == labels).sum().item()
                total_c += labels.size(0)
        print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}, Accuracy: {100*correct/total_c:.2f}%")

print("Student alone training done.")

Training student alone...
Epoch 5, Loss: 1.1840, Accuracy: 52.04%
Epoch 10, Loss: 0.8065, Accuracy: 54.07%
Epoch 15, Loss: 0.5625, Accuracy: 54.18%
Student alone training done.


In [ ]:
student_distilled_cifar = CIFARStudent()
optimizer_d = optim.Adam(student_distilled_cifar.parameters(), lr=0.001)
scheduler_d = optim.lr_scheduler.StepLR(optimizer_d, step_size=5, gamma=0.5)

temperature = 6.0   # higher temp = softer labels = more nuance transferred
alpha = 0.7         # 70% teacher, 30% real labels

print("Training distilled student...")
for epoch in range(15):
    student_distilled_cifar.train()
    total_loss = 0
    for images, labels in train_loader:
        optimizer_d.zero_grad()

        with torch.no_grad():
            teacher_output = teacher_cifar(images)

        student_output = student_distilled_cifar(images)

        hard_loss = criterion(student_output, labels)

        soft_loss = F.kl_div(
            F.log_softmax(student_output / temperature, dim=1),
            F.softmax(teacher_output / temperature, dim=1),
            reduction='batchmean'
        ) * (temperature ** 2)

        loss = alpha * soft_loss + (1 - alpha) * hard_loss
        loss.backward()
        optimizer_d.step()
        total_loss += loss.item()
    scheduler_d.step()

    if (epoch + 1) % 5 == 0:
        student_distilled_cifar.eval()
        correct = 0
        total_c = 0
        with torch.no_grad():
            for images, labels in test_loader:
                output = student_distilled_cifar(images)
                predicted = torch.argmax(output, dim=1)
                correct += (predicted == labels).sum().item()
                total_c += labels.size(0)
        print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}, Accuracy: {100*correct/total_c:.2f}%")

print("Distilled student training done.")

Training distilled student...
Epoch 5, Loss: 0.5856, Accuracy: 53.95%
Epoch 10, Loss: 0.4300, Accuracy: 55.05%
Epoch 15, Loss: 0.3677, Accuracy: 55.41%
Distilled student training done.


In [ ]:
def get_accuracy(mdl, loader):
    mdl.eval()
    correct = 0
    total_c = 0
    with torch.no_grad():
        for images, labels in loader:
            output = mdl(images)
            predicted = torch.argmax(output, dim=1)
            correct += (predicted == labels).sum().item()
            total_c += labels.size(0)
    return 100 * correct / total_c

teacher_acc = get_accuracy(teacher_cifar, test_loader)
alone_acc = get_accuracy(student_alone, test_loader)
distilled_acc = get_accuracy(student_distilled_cifar, test_loader)

torch.save(teacher_cifar.state_dict(), 'cifar_teacher.pth')
torch.save(student_alone.state_dict(), 'cifar_student_alone.pth')
torch.save(student_distilled_cifar.state_dict(), 'cifar_student_distilled.pth')

size_teacher = os.path.getsize('cifar_teacher.pth') / 1024
size_alone = os.path.getsize('cifar_student_alone.pth') / 1024
size_distilled = os.path.getsize('cifar_student_distilled.pth') / 1024

print("=" * 55)
print("     CIFAR-10 DISTILLATION RESULTS")
print("=" * 55)
print(f"Teacher:              {size_teacher:.2f} KB   {teacher_acc:.2f}%")
print(f"Student alone:        {size_alone:.2f} KB   {alone_acc:.2f}%")
print(f"Student distilled:    {size_distilled:.2f} KB   {distilled_acc:.2f}%")
print(f"Distillation gain:    +{distilled_acc - alone_acc:.2f}%")
print("=" * 55)

     CIFAR-10 DISTILLATION RESULTS
Teacher:              14992.60 KB   55.86%
Student alone:        3209.60 KB   54.18%
Student distilled:    3209.71 KB   55.41%
Distillation gain:    +1.23%


In [ ]:
import torch.nn.utils.prune as prune

# Work on a copy of your distilled student
pruned_student = CIFARStudent()
pruned_student.load_state_dict(student_distilled_cifar.state_dict())

# Collect all (module, weight_name) pairs we want to prune
parameters_to_prune = (
    (pruned_student.fc1, 'weight'),
    (pruned_student.fc2, 'weight'),
    (pruned_student.fc3, 'weight'),
)

# Global unstructured pruning — removes lowest 40% weights across ALL layers
prune.global_unstructured(
    parameters_to_prune,
    pruning_method=prune.L1Unstructured,
    amount=0.40,   # prune 40% of weights
)

# Check sparsity per layer
for name, module in pruned_student.named_modules():
    if isinstance(module, nn.Linear):
        total = module.weight.nelement()
        zeros = float(torch.sum(module.weight == 0))
        print(f"{name} sparsity: {100 * zeros / total:.1f}%")

fc1 sparsity: 40.5%
fc2 sparsity: 28.2%
fc3 sparsity: 20.9%


In [ ]:
# Make pruning permanent first — removes the mask, bakes zeros in
for module, name in parameters_to_prune:
    prune.remove(module, name)

# Fine-tune for 5 epochs — short, just recovering
optimizer_prune = optim.Adam(pruned_student.parameters(), lr=0.0003)  # smaller LR
criterion = nn.CrossEntropyLoss()

print("Fine-tuning pruned student...")
for epoch in range(5):
    pruned_student.train()
    total_loss = 0
    for images, labels in train_loader:
        optimizer_prune.zero_grad()
        output = pruned_student(images)
        loss = criterion(output, labels)
        loss.backward()
        optimizer_prune.step()
        total_loss += loss.item()

    pruned_student.eval()
    correct = 0
    total_c = 0
    with torch.no_grad():
        for images, labels in test_loader:
            output = pruned_student(images)
            predicted = torch.argmax(output, dim=1)
            correct += (predicted == labels).sum().item()
            total_c += labels.size(0)
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}, Accuracy: {100*correct/total_c:.2f}%")

print("Fine-tuning done.")

Fine-tuning pruned student...
Epoch 1, Loss: 0.8869, Accuracy: 55.37%
Epoch 2, Loss: 0.8328, Accuracy: 55.23%
Epoch 3, Loss: 0.7907, Accuracy: 55.41%
Epoch 4, Loss: 0.7519, Accuracy: 55.58%
Epoch 5, Loss: 0.7143, Accuracy: 55.63%
Fine-tuning done.


In [ ]:
# Dynamic quantization — simplest, works great for Linear layers
quantized_student = torch.quantization.quantize_dynamic(
    pruned_student,
    {nn.Linear},       # which layer types to quantize
    dtype=torch.qint8  # int8 = 4x smaller than float32
)

# Check accuracy after quantization
quantized_student.eval()
correct = 0
total_c = 0
with torch.no_grad():
    for images, labels in test_loader:
        output = quantized_student(images)
        predicted = torch.argmax(output, dim=1)
        correct += (predicted == labels).sum().item()
        total_c += labels.size(0)

quantized_acc = 100 * correct / total_c
print(f"Quantized accuracy: {quantized_acc:.2f}%")

/tmp/ipykernel_799/2041298308.py:2: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_student = torch.quantization.quantize_dynamic(


Quantized accuracy: 55.73%


In [ ]:
# Save all models
torch.save(student_distilled_cifar.state_dict(), 'cifar_distilled.pth')
torch.save(pruned_student.state_dict(),          'cifar_pruned.pth')
torch.save(quantized_student.state_dict(),       'cifar_quantized.pth')

# Get file sizes
def kb(path): return os.path.getsize(path) / 1024

# Get accuracies
distilled_acc  = get_accuracy(student_distilled_cifar, test_loader)
pruned_acc     = get_accuracy(pruned_student, test_loader)
# quantized_acc already computed above

print("=" * 60)
print("     FULL COMPRESSION PIPELINE — CIFAR-10")
print("=" * 60)
print(f"{'Model':<25} {'Size (KB)':>10} {'Accuracy':>10}")
print("-" * 60)
print(f"{'Distilled student':<25} {kb('cifar_distilled.pth'):>10.1f} {distilled_acc:>9.2f}%")
print(f"{'Pruned (40%) + fine-tune':<25} {kb('cifar_pruned.pth'):>10.1f} {pruned_acc:>9.2f}%")
print(f"{'Pruned + Quantized':<25} {kb('cifar_quantized.pth'):>10.1f} {quantized_acc:>9.2f}%")
print("=" * 60)
print(f"Total size reduction: {kb('cifar_distilled.pth')/kb('cifar_quantized.pth'):.1f}x smaller")
print(f"Total accuracy cost:  -{distilled_acc - quantized_acc:.2f}%")
print("=" * 60)

     FULL COMPRESSION PIPELINE — CIFAR-10
Model                      Size (KB)   Accuracy
------------------------------------------------------------
Distilled student             3209.6     55.41%
Pruned (40%) + fine-tune      3209.5     55.63%
Pruned + Quantized             807.8     55.73%
Total size reduction: 4.0x smaller
Total accuracy cost:  --0.32%


 **HAR — Human Activity Recognition
Full compression pipeline.**

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import os

# Download HAR dataset
os.makedirs('har_data', exist_ok=True)

# Fetch raw files
import urllib.request

base_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00240/"
files = [
    "UCI%20HAR%20Dataset.zip"
]

urllib.request.urlretrieve(
    base_url + "UCI%20HAR%20Dataset.zip",
    "har_data/har.zip"
)

import zipfile
with zipfile.ZipFile("har_data/har.zip", 'r') as z:
    z.extractall("har_data/")

print("Downloaded and extracted.")

Downloaded and extracted.


In [ ]:
def load_har():
    base = "har_data/UCI HAR Dataset/"

    X_train = np.loadtxt(base + "train/X_train.txt")
    y_train = np.loadtxt(base + "train/y_train.txt", dtype=int) - 1  # 0-indexed

    X_test = np.loadtxt(base + "test/X_test.txt")
    y_test = np.loadtxt(base + "test/y_test.txt", dtype=int) - 1

    # Convert to tensors
    X_train = torch.tensor(X_train, dtype=torch.float32)
    y_train = torch.tensor(y_train, dtype=torch.long)
    X_test  = torch.tensor(X_test,  dtype=torch.float32)
    y_test  = torch.tensor(y_test,  dtype=torch.long)

    train_ds = TensorDataset(X_train, y_train)
    test_ds  = TensorDataset(X_test,  y_test)

    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
    test_loader  = DataLoader(test_ds,  batch_size=64, shuffle=False)

    return train_loader, test_loader

train_loader, test_loader = load_har()
print("HAR loaded.")
print(f"Input features: 561")
print(f"Classes: 6 (walking, upstairs, downstairs, sitting, standing, laying)")

HAR loaded.
Input features: 561
Classes: 6 (walking, upstairs, downstairs, sitting, standing, laying)


In [ ]:
# Teacher — deep, wide, strong
class HARTeacher(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(561, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 6)
        )

    def forward(self, x):
        return self.net(x)


# Student — tiny, edge-ready
class HARStudent(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(561, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 6)
        )

    def forward(self, x):
        return self.net(x)


teacher = HARTeacher()
student_alone     = HARStudent()
student_distilled = HARStudent()

print("Teacher parameters:", sum(p.numel() for p in teacher.parameters()))
print("Student parameters:", sum(p.numel() for p in student_alone.parameters()))

Teacher parameters: 452742
Student parameters: 80582


In [ ]:
def train_model(model, loader, epochs, lr=0.001, label="Model"):
    optimizer  = optim.Adam(model.parameters(), lr=lr)
    scheduler  = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)
    criterion  = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for X, y in loader:
            optimizer.zero_grad()
            loss = criterion(model(X), y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        scheduler.step()

        if (epoch + 1) % 5 == 0:
            acc = get_accuracy(model, test_loader)
            print(f"[{label}] Epoch {epoch+1} | Loss: {total_loss/len(loader):.4f} | Acc: {acc:.2f}%")

def get_accuracy(model, loader):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for X, y in loader:
            pred = torch.argmax(model(X), dim=1)
            correct += (pred == y).sum().item()
            total   += y.size(0)
    return 100 * correct / total

print("Training teacher...")
train_model(teacher, train_loader, epochs=20, label="Teacher")
print(f"Teacher final accuracy: {get_accuracy(teacher, test_loader):.2f}%")

Training teacher...
[Teacher] Epoch 5 | Loss: 0.1099 | Acc: 89.96%
[Teacher] Epoch 10 | Loss: 0.0680 | Acc: 94.88%
[Teacher] Epoch 15 | Loss: 0.0467 | Acc: 94.06%
[Teacher] Epoch 20 | Loss: 0.0398 | Acc: 94.64%
Teacher final accuracy: 94.64%


In [ ]:
print("Training student alone...")
train_model(student_alone, train_loader, epochs=20, label="Student-Alone")
print(f"Student alone final accuracy: {get_accuracy(student_alone, test_loader):.2f}%")

Training student alone...
[Student-Alone] Epoch 5 | Loss: 0.1044 | Acc: 94.84%
[Student-Alone] Epoch 10 | Loss: 0.0562 | Acc: 94.27%
[Student-Alone] Epoch 15 | Loss: 0.0493 | Acc: 94.60%
[Student-Alone] Epoch 20 | Loss: 0.0405 | Acc: 94.74%
Student alone final accuracy: 94.74%


In [ ]:
def train_distilled(student, teacher, loader, epochs, temperature=4.0, alpha=0.7):
    optimizer = optim.Adam(student.parameters(), lr=0.001)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)
    criterion = nn.CrossEntropyLoss()
    teacher.eval()

    for epoch in range(epochs):
        student.train()
        total_loss = 0
        for X, y in loader:
            optimizer.zero_grad()

            with torch.no_grad():
                teacher_logits = teacher(X)

            student_logits = student(X)

            # Hard loss — real labels
            hard_loss = criterion(student_logits, y)

            # Soft loss — teacher's knowledge
            soft_loss = F.kl_div(
                F.log_softmax(student_logits  / temperature, dim=1),
                F.softmax(teacher_logits / temperature, dim=1),
                reduction='batchmean'
            ) * (temperature ** 2)

            loss = alpha * soft_loss + (1 - alpha) * hard_loss
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        scheduler.step()

        if (epoch + 1) % 5 == 0:
            acc = get_accuracy(student, test_loader)
            print(f"[Distilled] Epoch {epoch+1} | Loss: {total_loss/len(loader):.4f} | Acc: {acc:.2f}%")

print("Training distilled student...")
train_distilled(student_distilled, teacher, train_loader, epochs=20)
print(f"Distilled student final accuracy: {get_accuracy(student_distilled, test_loader):.2f}%")

Training distilled student...
[Distilled] Epoch 5 | Loss: 0.2115 | Acc: 94.98%
[Distilled] Epoch 10 | Loss: 0.0896 | Acc: 95.35%
[Distilled] Epoch 15 | Loss: 0.0664 | Acc: 94.77%
[Distilled] Epoch 20 | Loss: 0.0575 | Acc: 95.08%
Distilled student final accuracy: 95.08%


In [ ]:
import torch.nn.utils.prune as prune

pruned_student = HARStudent()
pruned_student.load_state_dict(student_distilled.state_dict())

# Prune 40% of weights globally
parameters_to_prune = (
    (pruned_student.net[0], 'weight'),
    (pruned_student.net[2], 'weight'),
    (pruned_student.net[4], 'weight'),
)

prune.global_unstructured(
    parameters_to_prune,
    pruning_method=prune.L1Unstructured,
    amount=0.40,
)

# Make pruning permanent
for module, name in parameters_to_prune:
    prune.remove(module, name)

# Check sparsity
for i, module in enumerate(pruned_student.net):
    if isinstance(module, nn.Linear):
        zeros = float(torch.sum(module.weight == 0))
        total = module.weight.nelement()
        print(f"Layer {i} sparsity: {100*zeros/total:.1f}%")

# Fine-tune to recover accuracy
print("Fine-tuning pruned student...")
train_model(pruned_student, train_loader, epochs=5, lr=0.0003, label="Pruned")
print(f"Pruned student accuracy: {get_accuracy(pruned_student, test_loader):.2f}%")

Layer 0 sparsity: 42.3%
Layer 2 sparsity: 21.5%
Layer 4 sparsity: 11.2%
Fine-tuning pruned student...
[Pruned] Epoch 5 | Loss: 0.0437 | Acc: 94.30%
Pruned student accuracy: 94.30%


In [ ]:
quantized_student = torch.quantization.quantize_dynamic(
    pruned_student,
    {nn.Linear},
    dtype=torch.qint8
)

quant_acc = get_accuracy(quantized_student, test_loader)
print(f"Quantized student accuracy: {quant_acc:.2f}%")

Quantized student accuracy: 93.99%


/tmp/ipykernel_799/2255432894.py:1: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_student = torch.quantization.quantize_dynamic(


In [ ]:
# Save models
torch.save(teacher.state_dict(),           'har_teacher.pth')
torch.save(student_alone.state_dict(),     'har_student_alone.pth')
torch.save(student_distilled.state_dict(), 'har_distilled.pth')
torch.save(pruned_student.state_dict(),    'har_pruned.pth')
torch.save(quantized_student.state_dict(), 'har_quantized.pth')

def kb(path): return os.path.getsize(path) / 1024

t_acc  = get_accuracy(teacher,           test_loader)
sa_acc = get_accuracy(student_alone,     test_loader)
sd_acc = get_accuracy(student_distilled, test_loader)
sp_acc = get_accuracy(pruned_student,    test_loader)

print("=" * 62)
print("        FULL EDGE AI COMPRESSION PIPELINE — HAR")
print("=" * 62)
print(f"{'Model':<28} {'Size (KB)':>10} {'Accuracy':>10}")
print("-" * 62)
print(f"{'Teacher':<28} {kb('har_teacher.pth'):>10.1f} {t_acc:>9.2f}%")
print(f"{'Student alone':<28} {kb('har_student_alone.pth'):>10.1f} {sa_acc:>9.2f}%")
print(f"{'Student distilled':<28} {kb('har_distilled.pth'):>10.1f} {sd_acc:>9.2f}%")
print(f"{'Pruned (40%)':<28} {kb('har_pruned.pth'):>10.1f} {sp_acc:>9.2f}%")
print(f"{'Pruned + Quantized':<28} {kb('har_quantized.pth'):>10.1f} {quant_acc:>9.2f}%")
print("=" * 62)
print(f"Compression ratio:  {kb('har_teacher.pth')/kb('har_quantized.pth'):.1f}x smaller than teacher")
print(f"Accuracy cost:      -{t_acc - quant_acc:.2f}% vs teacher")
print("=" * 62)

        FULL EDGE AI COMPRESSION PIPELINE — HAR
Model                         Size (KB)   Accuracy
--------------------------------------------------------------
Teacher                          1772.1     94.64%
Student alone                     317.9     94.74%
Student distilled                 317.8     95.08%
Pruned (40%)                      317.8     94.30%
Pruned + Quantized                 84.3     93.99%
Compression ratio:  21.0x smaller than teacher
Accuracy cost:      -0.64% vs teacher


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import os

# Download HAR dataset
os.makedirs('har_data', exist_ok=True)

import urllib.request
urllib.request.urlretrieve(
    "https://archive.ics.uci.edu/ml/machine-learning-databases/00240/UCI%20HAR%20Dataset.zip",
    "har_data/har.zip"
)

import zipfile
with zipfile.ZipFile("har_data/har.zip", 'r') as z:
    z.extractall("har_data/")

print("Downloaded and extracted successfully.")

Downloaded and extracted successfully.


In [ ]:
def load_har():
    base = "har_data/UCI HAR Dataset/"

    X_train = np.loadtxt(base + "train/X_train.txt")
    y_train = np.loadtxt(base + "train/y_train.txt", dtype=int) - 1

    X_test = np.loadtxt(base + "test/X_test.txt")
    y_test = np.loadtxt(base + "test/y_test.txt", dtype=int) - 1

    X_train = torch.tensor(X_train, dtype=torch.float32)
    y_train = torch.tensor(y_train, dtype=torch.long)
    X_test  = torch.tensor(X_test,  dtype=torch.float32)
    y_test  = torch.tensor(y_test,  dtype=torch.long)

    train_ds = TensorDataset(X_train, y_train)
    test_ds  = TensorDataset(X_test,  y_test)

    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
    test_loader  = DataLoader(test_ds,  batch_size=64, shuffle=False)

    return train_loader, test_loader

train_loader, test_loader = load_har()
print("HAR loaded.")
print(f"Input features: 561")
print(f"Classes: 6 (walking, upstairs, downstairs, sitting, standing, laying)")

HAR loaded.
Input features: 561
Classes: 6 (walking, upstairs, downstairs, sitting, standing, laying)


In [ ]:
class HARTeacher(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(561, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 6)
        )

    def forward(self, x):
        return self.net(x)


class HARStudent(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(561, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 6)
        )

    def forward(self, x):
        return self.net(x)


teacher          = HARTeacher()
student_alone    = HARStudent()
student_distilled = HARStudent()

print("Teacher parameters:", sum(p.numel() for p in teacher.parameters()))
print("Student parameters:", sum(p.numel() for p in student_alone.parameters()))

Teacher parameters: 452742
Student parameters: 80582


In [ ]:
def get_accuracy(model, loader):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for X, y in loader:
            pred = torch.argmax(model(X), dim=1)
            correct += (pred == y).sum().item()
            total   += y.size(0)
    return 100 * correct / total

def train_model(model, loader, epochs, lr=0.001, label="Model"):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for X, y in loader:
            optimizer.zero_grad()
            loss = criterion(model(X), y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        scheduler.step()

        if (epoch + 1) % 5 == 0:
            acc = get_accuracy(model, test_loader)
            print(f"[{label}] Epoch {epoch+1} | Loss: {total_loss/len(loader):.4f} | Acc: {acc:.2f}%")

print("Training teacher...")
train_model(teacher, train_loader, epochs=20, label="Teacher")
print(f"Teacher final accuracy: {get_accuracy(teacher, test_loader):.2f}%")

Training teacher...
[Teacher] Epoch 5 | Loss: 0.1199 | Acc: 92.23%
[Teacher] Epoch 10 | Loss: 0.0618 | Acc: 94.98%
[Teacher] Epoch 15 | Loss: 0.0510 | Acc: 94.13%
[Teacher] Epoch 20 | Loss: 0.0398 | Acc: 95.05%
Teacher final accuracy: 95.05%


In [ ]:
print("Training student alone...")
train_model(student_alone, train_loader, epochs=20, label="Student-Alone")
print(f"Student alone final accuracy: {get_accuracy(student_alone, test_loader):.2f}%")

Training student alone...
[Student-Alone] Epoch 5 | Loss: 0.0856 | Acc: 92.23%
[Student-Alone] Epoch 10 | Loss: 0.0568 | Acc: 91.25%
[Student-Alone] Epoch 15 | Loss: 0.0433 | Acc: 94.30%
[Student-Alone] Epoch 20 | Loss: 0.0401 | Acc: 94.88%
Student alone final accuracy: 94.88%


In [ ]:
def train_distilled(student, teacher, loader, epochs, temperature=4.0, alpha=0.7):
    optimizer = optim.Adam(student.parameters(), lr=0.001)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)
    criterion = nn.CrossEntropyLoss()
    teacher.eval()

    for epoch in range(epochs):
        student.train()
        total_loss = 0
        for X, y in loader:
            optimizer.zero_grad()

            with torch.no_grad():
                teacher_logits = teacher(X)

            student_logits = student(X)

            hard_loss = criterion(student_logits, y)

            soft_loss = F.kl_div(
                F.log_softmax(student_logits / temperature, dim=1),
                F.softmax(teacher_logits / temperature, dim=1),
                reduction='batchmean'
            ) * (temperature ** 2)

            loss = alpha * soft_loss + (1 - alpha) * hard_loss
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        scheduler.step()

        if (epoch + 1) % 5 == 0:
            acc = get_accuracy(student, test_loader)
            print(f"[Distilled] Epoch {epoch+1} | Loss: {total_loss/len(loader):.4f} | Acc: {acc:.2f}%")

print("Training distilled student...")
train_distilled(student_distilled, teacher, train_loader, epochs=20)
print(f"Distilled student final accuracy: {get_accuracy(student_distilled, test_loader):.2f}%")

Training distilled student...
[Distilled] Epoch 5 | Loss: 0.2251 | Acc: 93.69%
[Distilled] Epoch 10 | Loss: 0.0889 | Acc: 95.25%
[Distilled] Epoch 15 | Loss: 0.0634 | Acc: 95.22%
[Distilled] Epoch 20 | Loss: 0.0551 | Acc: 95.32%
Distilled student final accuracy: 95.32%


In [ ]:
student_distilled_v2 = HARStudent()

train_distilled(
    student_distilled_v2,
    teacher,
    train_loader,
    epochs=20,
    temperature=2.0,  # sharper — less soft
    alpha=0.3         # trust teacher less, trust real labels more
)

print(f"Distilled v2 accuracy: {get_accuracy(student_distilled_v2, test_loader):.2f}%")

[Distilled] Epoch 5 | Loss: 0.1140 | Acc: 94.60%
[Distilled] Epoch 10 | Loss: 0.0537 | Acc: 94.16%
[Distilled] Epoch 15 | Loss: 0.0429 | Acc: 94.88%
[Distilled] Epoch 20 | Loss: 0.0377 | Acc: 95.18%
Distilled v2 accuracy: 95.18%


In [ ]:
 import torch.nn.utils.prune as prune

pruned_student = HARStudent()
pruned_student.load_state_dict(student_distilled_v2.state_dict())

parameters_to_prune = (
    (pruned_student.net[0], 'weight'),
    (pruned_student.net[2], 'weight'),
    (pruned_student.net[4], 'weight'),
)

prune.global_unstructured(
    parameters_to_prune,
    pruning_method=prune.L1Unstructured,
    amount=0.40,
)

for module, name in parameters_to_prune:
    prune.remove(module, name)

for i, module in enumerate(pruned_student.net):
    if isinstance(module, nn.Linear):
        zeros = float(torch.sum(module.weight == 0))
        total = module.weight.nelement()
        print(f"Layer {i} sparsity: {100*zeros/total:.1f}%")

print("Fine-tuning pruned student...")
train_model(pruned_student, train_loader, epochs=5, lr=0.0003, label="Pruned")
print(f"Pruned student accuracy: {get_accuracy(pruned_student, test_loader):.2f}%")

Layer 0 sparsity: 42.3%
Layer 2 sparsity: 21.5%
Layer 4 sparsity: 12.5%
Fine-tuning pruned student...
[Pruned] Epoch 5 | Loss: 0.0427 | Acc: 94.30%
Pruned student accuracy: 94.30%


In [ ]:
quantized_student = torch.quantization.quantize_dynamic(
    pruned_student,
    {nn.Linear},
    dtype=torch.qint8
)

quant_acc = get_accuracy(quantized_student, test_loader)
print(f"Quantized student accuracy: {quant_acc:.2f}%")

Quantized student accuracy: 94.40%


/tmp/ipykernel_1391/2255432894.py:1: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_student = torch.quantization.quantize_dynamic(


In [ ]:
torch.save(teacher.state_dict(),              'har_teacher.pth')
torch.save(student_alone.state_dict(),        'har_student_alone.pth')
torch.save(student_distilled_v2.state_dict(), 'har_distilled.pth')
torch.save(pruned_student.state_dict(),       'har_pruned.pth')
torch.save(quantized_student.state_dict(),    'har_quantized.pth')

def kb(path): return os.path.getsize(path) / 1024

t_acc  = get_accuracy(teacher, test_loader)
sa_acc = get_accuracy(student_alone, test_loader)
sd_acc = get_accuracy(student_distilled_v2, test_loader)
sp_acc = get_accuracy(pruned_student, test_loader)

print("=" * 62)
print("     FULL EDGE AI COMPRESSION PIPELINE — HAR")
print("=" * 62)
print(f"{'Model':<28} {'Size (KB)':>10} {'Accuracy':>10}")
print("-" * 62)
print(f"{'Teacher':<28} {kb('har_teacher.pth'):>10.1f} {t_acc:>9.2f}%")
print(f"{'Student alone':<28} {kb('har_student_alone.pth'):>10.1f} {sa_acc:>9.2f}%")
print(f"{'Student distilled':<28} {kb('har_distilled.pth'):>10.1f} {sd_acc:>9.2f}%")
print(f"{'Pruned 40%':<28} {kb('har_pruned.pth'):>10.1f} {sp_acc:>9.2f}%")
print(f"{'Pruned + Quantized':<28} {kb('har_quantized.pth'):>10.1f} {94.40:>9.2f}%")
print("=" * 62)
print(f"Compression ratio: {kb('har_teacher.pth')/kb('har_quantized.pth'):.1f}x smaller than teacher")
print(f"Accuracy cost:     -{t_acc - 94.40:.2f}% vs teacher")
print("=" * 62)

     FULL EDGE AI COMPRESSION PIPELINE — HAR
Model                         Size (KB)   Accuracy
--------------------------------------------------------------
Teacher                          1772.1     95.05%
Student alone                     317.9     18.05%
Student distilled                 317.8     95.18%
Pruned 40%                        317.8     94.30%
Pruned + Quantized                 84.3     94.40%
Compression ratio: 21.0x smaller than teacher
Accuracy cost:     -0.65% vs teacher
